# **Agentes en el Arcade Learning Environment (ALE): Space Invaders**

# **1. El Arcade Learning Environment (ALE)**

> **Nota sobre el uso de inteligencia artificial.** Para apoyar la comprensión de términos y conceptos relevantes para este laboratorio, se utilizó GPT-5.6 Sol como herramienta de consulta mediante diversas preguntas y solicitudes de explicación.

**¿Qué es el Arcade Learning Environment (ALE) y qué problema resuelve? ¿Qué relación tiene con el emulador Stella y con los juegos originales de Atari 2600?**

El Arcade Learning Environment (ALE) es una plataforma para evaluar agentes de inteligencia artificial con juegos de Atari 2600. Su propósito es convertir cada juego en un entorno de aprendizaje por refuerzo con una interfaz común, donde el agente recibe una observación, selecciona una acción y obtiene una recompensa junto con una indicación de si el episodio terminó. De esta forma, el mismo algoritmo puede probarse en juegos con objetivos y dinámicas diferentes.

ALE resuelve principalmente la falta de un estándar experimental. Por ejemplo, si cada investigador utilizara un emulador y una definición de recompensa diferentes, sus resultados no serían directamente comparables. Al utilizar la misma interfaz y los mismos juegos se busca que las diferencias observadas dependan del agente y no de una implementación particular del entorno.

**Stella** es el emulador que reproduce el hardware de la consola y ejecuta las ROM, las cuales contienen el programa y los datos de los cartuchos originales. ALE se construye sobre Stella y agrega la capa que permite obtener imágenes o memoria RAM, enviar acciones del control y extraer la puntuación y la condición de fin. Básicamente, la relación es: se tiene el Rom del juego, luego Stella lo ejecuta, ALE lo convierte en un entorno medible y Gymnasium permite utilizarlo en Python.

**¿Qué diferencias existen entre las variantes de un mismo juego en ALE, por ejemplo ALE/SpaceInvaders-v5 frente a versiones con sufijos como -ram o parámetros como frameskip, repeat_action_probability y full_action_space?**

Las variantes de Space Invaders ejecutan la misma ROM, pero cambian las condiciones bajo las que interactúa el agente. ALE/SpaceInvaders-v5 es la configuración moderna de referencia y utiliza una imagen RGB, frameskip=4, repeat_action_probability=0.25 y el conjunto mínimo de seis acciones útiles. El prefijo ALE/ pertenece al esquema actual de nombres.

- **-ram u obs_type="ram":** cambia la observación visual por los 128 bytes de memoria de la consola. El sufijo pertenece al esquema histórico; en la interfaz actual puede solicitarse con gymnasium.make("ALE/SpaceInvaders-v5", obs_type="ram"). La dinámica y la recompensa se mantienen, lo que cambia es la información que recibe el agente.
- **frameskip:** indica durante cuántos cuadros se mantiene una acción antes de pedir la siguiente. Un entero establece una cantidad fija y una tupla permite seleccionarla aleatoriamente dentro de un intervalo.
- **repeat_action_probability:** establece la probabilidad de ignorar la acción nueva y repetir la anterior. Estas "sticky actions" agregan incertidumbre, buscando que el agente observe el juego en lugar de limitarse a memorizar una secuencia fija.
- **full_action_space:** con False se utilizan únicamente las seis acciones que tienen sentido en Space Invaders. Con True se exponen las 18 combinaciones legales del control de Atari, aunque varias sean redundantes, de tal forma que también aumenta el espacio que tendría que explorar el agente.

**¿Por qué es importante el concepto de frame skipping (salto de frames) en los entornos de Atari, y cómo afecta la velocidad de simulación y el aprendizaje de un agente?**

El frame skipping indica cuántos cuadros del juego se avanzan manteniendo la misma acción. Por ejemplo, con frameskip=4 el agente toma una decisión, el entorno la ejecuta durante cuatro cuadros y luego devuelve la siguiente observación. Por lo tanto, un paso de Gymnasium no equivale necesariamente a un solo cuadro del emulador.

Su propósito es reducir la frecuencia de decisión y el costo del aprendizaje. El emulador continúa actualizando el juego, pero la política se consulta menos veces y se almacenan menos transiciones para cubrir la misma duración. Esto también resulta acorde a acciones como moverse a la izquierda, ya que deben mantenerse durante varios cuadros para producir un desplazamiento visible.

Sin embargo, un salto demasiado alto reduce la capacidad de reaccionar, pues el agente podría moverse más de lo necesario o no responder a tiempo ante un proyectil. Además, cambia la escala de las métricas: con un salto de cuatro, mil pasos representan aproximadamente cuatro mil cuadros. Finalmente, no debe confundirse con repeat_action_probability, ya que el frame skipping repite una acción de forma intencional, mientras que el segundo parámetro introduce incertidumbre sobre cuál acción se ejecuta.

**Investigue brevemente el juego Space Invaders: mecánica básica, objetivo y cómo se traduce la puntuación del juego original a la señal de recompensa (reward) del entorno.**

En Space Invaders se controla un cañón que puede moverse horizontalmente y disparar desde la parte inferior de la pantalla. El objetivo es destruir los 36 invasores antes de que alcancen la Tierra, evitando sus proyectiles y utilizando los escudos como protección temporal. Conforme se eliminan enemigos, los restantes se mueven con mayor velocidad. La partida termina cuando se pierden las tres oportunidades o cuando los invasores llegan a la parte inferior; si se elimina una formación completa, aparece otra más cerca de la Tierra.

La puntuación depende del objetivo destruido. Los enemigos de las seis filas valen 5, 10, 15, 20, 25 y 30 puntos, por lo que una formación completa suma 630. Adicional, una nave de comando aparece ocasionalmente en la parte superior y vale 200 puntos en el modo normal. El marcador original muestra hasta 9999 y luego vuelve a comenzar.

ALE utiliza el cambio en esa puntuación como recompensa. Por ejemplo, si se destruye un invasor de 20 puntos, el paso produce un reward de 20; moverse, fallar un disparo o sobrevivir sin aumentar el marcador produce normalmente 0. Asimismo, ALE corrige el reinicio del marcador después de 9999 para no perder la recompensa acumulada. Esto implica que no existe una bonificación adicional por sobrevivir ni un castigo negativo directo por perder una oportunidad, de tal forma que el retorno del episodio representa los puntos obtenidos y no necesariamente cuánto tiempo logró sobrevivir el agente, que no es lo que interesa, ni en el juego ni en el proyecto que se desarrollará.